# Protein interface analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davismerighi-max/interface-analysis/blob/main/notebooks/interface_analysis_colab.ipynb)

Give it a refined structure (`.pdb` / `.cif`, or a PDB entry code) and it measures every
interface: which residues are buried and by how much, the salt bridges, hydrogen bonds,
hydrophobic contacts, disulfides, metal bridges and stacking that hold the subunits together,
how well the surfaces fit, what bound cofactors touch, and which parts of the model could be
stable assemblies on their own. Then it draws the figures and hands you everything as a zip.

### How to use it — no programming needed

1. **Fill in the settings form ①** below. Every option is there, and nothing runs while you
   edit it.
2. **Runtime → Run all** (or `Ctrl`+`F9`).

If you ticked an upload, its **Choose files** button appears at the top of box ② straight
away; after that the analysis runs on its own and the results appear under box ②. A zip with
everything, sorted into one folder per figure, downloads at the end.

To try it, just press *Run all*: the form is preset to the example entry 2TSC.

Code and documentation: https://github.com/davismerighi-max/interface-analysis

In [ ]:
#@title ① Settings — fill in, then choose Runtime → Run all { display-mode: "form" }
#@markdown ### Structure
#@markdown Type a PDB entry code, **or** tick *upload my structure* (the code is then ignored and an upload button appears when the run starts). A file you already dropped into the Files panel on the left can be typed instead of a code, e.g. `my_model.cif`.
pdb_code = "2TSC"  #@param {type:"string"}
upload_my_structure = False  #@param {type:"boolean"}
#@markdown Resolution of your map or data in Å. Leave 0 to read it from the file header; if it is not there either, bonds are reported as tentative.
resolution = 0  #@param {type:"number"}
#@markdown ---
#@markdown ### Proteins (optional): one per line, `name  colour  sequence`
#@markdown e.g. `YfdQ  #FF00F7  MDKSAIEAIQDSSAAAAK…`. Name without spaces; colour as `#RRGGBB` or a name such as `magenta`; paste the whole sequence. Proteins are recognised in the model **by sequence**, so order does not matter and the full-length sequence is fine. Empty lines are ignored; anything that does not match runs with defaults and the report says why. More than six proteins: tick *upload a FASTA* (headers like `>YfdQ_#FF00F7`).
protein_1 = ""  #@param {type:"string"}
protein_2 = ""  #@param {type:"string"}
protein_3 = ""  #@param {type:"string"}
protein_4 = ""  #@param {type:"string"}
protein_5 = ""  #@param {type:"string"}
protein_6 = ""  #@param {type:"string"}
upload_protein_fasta = False  #@param {type:"boolean"}
#@markdown ---
#@markdown ### Conservation (optional)
#@markdown Tick to upload one alignment per protein (aligned FASTA, A3M from ColabFold/HHblits, Clustal or Stockholm; the **first** sequence must be your protein). Without it the footprints have no conservation track.
upload_alignments = False  #@param {type:"boolean"}
#@markdown ---
#@markdown ### Analysis
#@markdown Shape complementarity is the slow part: about a minute for a dimer, ten minutes or more for ~30 chains. Untick for a quick first look (the packing figure is then left out).
shape_complementarity = True  #@param {type:"boolean"}
find_assemblies = True  #@param {type:"boolean"}
#@markdown ---
#@markdown ### Figures
residues_per_line = 350  #@param {type:"integer"}
draw_3D_renders = True  #@param {type:"boolean"}
#@markdown The renders show three views: from the side, tilted, and from the top. These set the tilted one (elevation 0 = side, 90 = top) and the rotation about the symmetry axis.
tilted_view_elevation = 62  #@param {type:"number"}
view_rotation = 20  #@param {type:"number"}
#@markdown ---
#@markdown ### Re-running
#@markdown Untick to redraw the figures of the SAME model with new figure settings, without measuring again (much faster).
measure_again = True  #@param {type:"boolean"}
download_zip_at_the_end = True  #@param {type:"boolean"}
print("settings stored")

In [ ]:
#@title ② Run everything { display-mode: "form" }
#@markdown Nothing to set here: this box uses the settings above, runs every step and shows the results.
import os, sys, re, time, glob, shutil, subprocess, base64
from IPython.display import display, Image, Markdown, HTML

CONTENT = "/content"
REPO = CONTENT + "/interface-analysis"
os.makedirs(CONTENT, exist_ok=True)
os.chdir(CONTENT)
try:
    from google.colab import files as _colab_files
except Exception:
    _colab_files = None


def _upload(what):
    if _colab_files is None:
        raise SystemExit(f"Uploading {what} needs Google Colab.")
    display(Markdown(f"**Choose {what}:**"))
    got = _colab_files.upload()
    if not got:
        raise SystemExit(f"No {what} chosen: untick the upload in the settings or try again.")
    return got


# ---- 1. files first, so you can walk away once they are chosen
UP_MODEL = _upload("your structure file (.pdb / .cif)") if upload_my_structure else {}
UP_FASTA = _upload("the protein FASTA file") if upload_protein_fasta else {}
UP_MSA = _upload("the alignment file(s) - one per protein") if upload_alignments else {}

# ---- 2. libraries and code, once per session
if not os.path.exists(REPO + "/01_interface_analysis.py"):
    print("installing libraries and fetching the code (about a minute) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biopython", "pydssp", "scikit-image"], check=True)
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/davismerighi-max/interface-analysis.git", REPO],
                   check=True)

# ---- 3. the structure
if UP_MODEL:
    _name, _data = list(UP_MODEL.items())[-1]
    MODEL = os.path.basename(_name).replace(" ", "_")
    _src_bytes = _data
else:
    _c = pdb_code.strip()
    if not _c:
        raise SystemExit("Settings: type a PDB code, or tick 'upload my structure'.")
    if _c.lower().endswith((".pdb", ".cif", ".ent", ".mmcif")):        # from the Files panel
        _p = _c if os.path.isabs(_c) else os.path.join(CONTENT, _c)
        if not os.path.exists(_p):
            raise SystemExit(f"Settings: there is no file called {_c} in the Files panel.")
        MODEL = os.path.basename(_p).replace(" ", "_")
        _src_bytes = open(_p, "rb").read()
    else:
        import urllib.request
        _src_bytes = None
        for _ext in (".pdb", ".cif"):            # very large entries exist only as mmCIF
            try:
                with urllib.request.urlopen(
                        f"https://files.rcsb.org/download/{_c.lower()}{_ext}") as _r:
                    _src_bytes = _r.read()
                MODEL = _c.lower() + _ext
                break
            except Exception:
                continue
        if _src_bytes is None:
            raise SystemExit(f"Could not download PDB entry {_c.upper()}: check the code.")

# each model works in its own folder, so results of different models never mix
RUN = os.path.join(CONTENT, "runs", os.path.splitext(MODEL)[0])
_reuse = (not measure_again) and os.path.exists(os.path.join(RUN, "run_config.json"))
if not _reuse:
    if os.path.isdir(RUN):
        shutil.rmtree(RUN)
    os.makedirs(RUN)
    open(os.path.join(RUN, MODEL), "wb").write(_src_bytes)
os.chdir(RUN)
print("model:", MODEL, "| working folder:", RUN,
      "| measurements reused" if _reuse else "")

# ---- 4. protein names, colours and sequences -> proteins.fasta
PROTEINS = None
if UP_FASTA:
    open("proteins.fasta", "wb").write(list(UP_FASTA.values())[-1])
    PROTEINS = "proteins.fasta"
else:
    from matplotlib.colors import to_hex
    _rows = []
    for _i in range(1, 7):
        _line = str(globals().get(f"protein_{_i}", "")).strip()
        if not _line:
            continue
        _tok = _line.replace(",", " ").split()
        _name, _col, _seq = _tok[0], None, _tok[1:]
        if _seq:
            try:
                _col = to_hex(_seq[0]); _seq = _seq[1:]
            except ValueError:
                pass
        _seq = re.sub(r"[^A-Za-z]", "", "".join(_seq)).upper()
        if not _seq:
            print(f"protein line {_i}: no sequence after the name - skipped")
            continue
        _rows.append(f">{_name}{'_' + _col if _col else ''}\n{_seq}\n")
    if _rows:
        open("proteins.fasta", "w").write("".join(_rows))
        PROTEINS = "proteins.fasta"

# ---- 5. alignments
MSAS = []
for _i, (_n, _d) in enumerate(UP_MSA.items(), 1):
    _safe = "".join(c if c.isalnum() or c in "._-" else "_" for c in os.path.basename(_n))
    open(f"msa_{_i}_{_safe}", "wb").write(_d)
    MSAS.append(f"msa_{_i}_{_safe}")


# ---- 6. the steps
def step(label, script, args):
    """Run one step; one progress line, the full log only if it stops."""
    t0 = time.time()
    print(f"{label} ...", end=" ", flush=True)
    p = subprocess.run([sys.executable, f"{REPO}/{script}", *args],
                       capture_output=True, text=True)
    if p.returncode:
        print(f"STOPPED after {time.time() - t0:.0f} s")
        print("\n".join((p.stdout + p.stderr).splitlines()[-25:]))
    else:
        print(f"done ({time.time() - t0:.0f} s)")
    return p.returncode


_ok = True
if not _reuse:
    a1 = [MODEL]
    if resolution and resolution > 0:
        a1 += ["--resolution", f"{resolution:g}"]
    a1 += ["--shape-workers", "2"] if shape_complementarity else ["--no-shape"]
    if PROTEINS:
        a1 += ["--proteins", PROTEINS]
    _ok = step("1/5  measuring the interfaces", "01_interface_analysis.py", a1) == 0
    _ok = _ok and step("2/5  tables and statistics", "02_derive_tables.py", [MODEL]) == 0
if _ok:
    a3 = [MODEL, "--per-line", str(residues_per_line)]
    a3 += (["--render-view", f"{tilted_view_elevation:g},{view_rotation:g}"]
           if draw_3D_renders else ["--no-render"])
    if MSAS:
        a3 += ["--msa", ",".join(MSAS)]
    step("3/5  figures", "03_make_figures.py", a3)
    if find_assemblies:
        step("4/5  candidate assemblies", "05_assemblies.py",
             [MODEL] + (["--render"] if draw_3D_renders else []))
    step("5/5  sorting the results", "06_organise_results.py", [MODEL])

# ---- 7. results
if os.path.exists("run_report.txt"):
    display(Markdown("## Run report"))
    print(open("run_report.txt", encoding="utf-8").read())
if not _ok:
    raise SystemExit("A step stopped: the report above says which and why "
                     "(details in run_errors.log).")


def _show(paths, title):
    paths = [p for p in paths if os.path.exists(p)]
    if paths:
        display(Markdown("## " + title))
        for p in paths:
            display(Markdown(f"`{p}`"))
            display(Image(filename=p, width=1000))


_show(["fig_render_proteins.png", "fig_render_budget.png", "fig_render_interactions.png"],
      "3D renders")
_show(sorted(glob.glob("fig*_footprints.png")), "Footprints")
_show(["fig_inventory.png", "fig_area_partition.png", "fig_chemistry.png", "fig_packing.png",
       "fig_architecture.png", "fig_contact_maps.png", "figS1_contact_maps_all.png"]
      + sorted(glob.glob("fig_interaction_networks*.png")), "Interfaces")
if os.path.exists("assembly_candidates.csv"):
    import pandas as pd
    pd.set_option("display.max_colwidth", 200)
    display(Markdown("## Candidate assemblies"))
    display(pd.read_csv("assembly_candidates.csv")[
        ["rank", "class", "kept_pct", "n_chains", "composition",
         "disulfides_inside", "disulfides_across_boundary", "reason"]])
    _show(["fig_assemblies.png"], "")

_base = os.path.splitext(MODEL)[0] + "_results"
if os.path.isdir(_base):
    _zip = shutil.make_archive(os.path.join(CONTENT, _base), "zip", RUN, _base)
    print("\nall results, one folder per figure:", _zip)
    if download_zip_at_the_end and _colab_files is not None:
        _colab_files.download(_zip)

---
### Reading three of the numbers

**Buried area** is computed pairwise in isolation and reported halved (the interface area,
not the sum of both footprints): PISA's convention, so directly comparable to it.

**Shape complementarity** reads about **0.10 below** the original `sc` program (trypsin–BPTI
0.647; obligate histone heterodimers 0.598–0.620; published ≈ 0.70), so the class names on the
figure are pessimistic. Use S_c to rank interfaces within one run.

**ΔG** is the Eisenberg–McLachlan solvation term (20 amino acids only). Negative = hydrophobic
burial pays; positive = the interface is held by hydrogen bonds and salt bridges instead. It is
not a binding free energy.

### If something goes wrong
Read the run report under box ②: each item says *ok*, *DEFAULT*, *WARNING*, *SKIPPED* or
*FAILED*, with the reason. A failure in one figure never stops the others. Very large
assemblies may exceed the free Colab machine's memory — run them locally or on a cluster; the
code is identical.